# Chapter 6 &mdash; Language Equivalence by Lock-Step Search

**Concept 4 of the Chapter 6 decomposition:** *Language Equivalence Checking by Lock-Step Search, with Counterexamples*

Concurrent DFS over state pairs; the first pair whose accept-status disagrees is a counterexample.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Equivalence-Checking/Concept-Language-Equivalence-Checking.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


To decide whether two DFA accept the same language, **walk them together**. Start at
$(q_{0,1}, q_{0,2})$; from each visited pair, follow every symbol in both machines at
once.

* if some reachable pair has **differing finality**, the path that reached it is a
  **counterexample string** &mdash; report it;
* if the search closes with no such pair, the languages are **equal**.

Termination is immediate: there are finitely many pairs. Jove's
`langeq_dfa(D1, D2, gen_counterex=True)` prints the witnessing path.

## 2. Definitions

### Three machines: two equal, one subtly different

In [ ]:
A = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
B = md2mc('''DFA
IF : 0 -> P
IF : 1 -> Q
P  : 0 -> IF
P  : 1 -> R
Q  : 0 -> R
Q  : 1 -> IF
R  : 0 -> Q
R  : 1 -> P
''')
C = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> Od      !! differs: 1s flip the parity too
Od : 0 -> IF
Od : 1 -> IF
''')

### The lock-step walk, written out, returning the witness string

In [ ]:
def lockstep(D1, D2):
    from collections import deque
    start = (D1["q0"], D2["q0"])
    seen, dq = {start: ''}, deque([start])
    while dq:
        p = dq.popleft(); w = seen[p]
        if (p[0] in D1["F"]) != (p[1] in D2["F"]):
            return w                       # counterexample
        for ch in sorted(D1["Sigma"]):
            n = (step_dfa(D1, p[0], ch), step_dfa(D2, p[1], ch))
            if n not in seen:
                seen[n] = w + ch; dq.append(n)
    return None                            # equivalent

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;3.&nbsp;Pruning Unreachable States](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Pruning-Unreachable/Concept-Pruning-Unreachable.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;5.&nbsp;Isomorphism = Language Equivalence + Equal State Count](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Isomorphism-Vs-Equivalence/Concept-Isomorphism-Vs-Equivalence.ipynb)&nbsp;&rarr;

---

## 3. Tests

Wait &mdash; `B` is not equal to `A`. The walk finds the shortest witness.

In [ ]:
w = lockstep(A, B)
print("witness :", repr(w))
if w is not None:
    print("  A accepts %r ? %s" % (w, accepts_dfa(A, w)))
    print("  B accepts %r ? %s" % (w, accepts_dfa(B, w)))
    assert accepts_dfa(A, w) != accepts_dfa(B, w)
print("langeq_dfa agrees :", langeq_dfa(A, B))
assert (w is None) == langeq_dfa(A, B)

A genuinely equivalent pair closes the search with no witness.

In [ ]:
A2 = md2mc('''DFA
IF : 0 -> X
IF : 1 -> IF
X  : 0 -> Y
X  : 1 -> X
Y  : 0 -> X       !! Y behaves exactly like IF's partner
Y  : 1 -> Y
''')
print("A vs A itself :", lockstep(A, A), langeq_dfa(A, A))
assert lockstep(A, A) is None and langeq_dfa(A, A)

And `C` differs from `A` almost immediately.

In [ ]:
w = lockstep(A, C)
print("witness :", repr(w), " len", len(w))
print("  A:", accepts_dfa(A, w), "  C:", accepts_dfa(C, w))
assert not langeq_dfa(A, C)

Jove prints the visited pairs when you ask for a counterexample.

In [ ]:
print(langeq_dfa(A, C, gen_counterex=True))

Termination: at most $|Q_1|\cdot|Q_2|$ pairs, so the walk always finishes.

In [ ]:
print("pair space for A vs B : %d x %d = %d"
      % (len(A["Q"]), len(B["Q"]), len(A["Q"]) * len(B["Q"])))

## 4. Exercises


1. Modify `lockstep` to return **all** shortest witnesses. How many are there?
2. Why must both DFA be **total** for the walk to be sound?
3. Compare the cost of `langeq_dfa` with "minimize both, then `iso_dfa`".

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6/Concept-Language-Equivalence-Checking')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')